# Combine microns1412 and visp_deltalakes with Taxonomy records

Reads delta tables from microns1412 and visp_deltalakes (same sources as yy05),
combines them, creates `Taxonomy` records for each classification system, and adds
`taxonomy` columns to `cluster`, `clustermembership`, and `celltoclustermapping` tables.
Writes the result to `/scratch/combined_with_taxonomies/`.

Three taxonomies:
- `tasic_2018_visp_scrnaseq` — T-type taxonomy from Tasic et al. 2018
- `visp_met_types` — MET-type taxonomy for VISp
- `minnie65` — Minnie65 EM clustering

In [1]:
import os
import sys

import pyarrow as pa
import polars as pl
from deltalake import DeltaTable, write_deltalake

In [2]:
sys.path.append(str("/root/capsule/src/"))

from connects_common_connectivity.models import Taxonomy
from connects_common_connectivity.arrow_utils import build_arrow_schema, models_to_table, attach_linkml_metadata

In [3]:
CODEOCEAN_MICRONS = "/data/microns1412"
CODEOCEAN_VISP = "/data/visp_deltalakes"
LOCAL_MICRONS = "../data/microns1412"
LOCAL_VISP = "../data/visp_deltalakes"

MICRONS_ROOT = CODEOCEAN_MICRONS if os.path.exists(CODEOCEAN_MICRONS) else LOCAL_MICRONS
VISP_ROOT = CODEOCEAN_VISP if os.path.exists(CODEOCEAN_VISP) else LOCAL_VISP
OUTPUT_ROOT = "/scratch/combined_with_taxonomies"

print(f"Microns root: {MICRONS_ROOT}")
print(f"VISP root:    {VISP_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

Input root:  ../data/combined_datasets
Output root: /scratch/combined_with_taxonomies


In [ ]:
def discover_delta_tables(root: str) -> dict[str, str]:
    """
    Walk *root* and return {relative_table_path: absolute_table_path}
    for every directory that contains a `_delta_log/` subdirectory.
    """
    root = os.path.abspath(root)
    tables = {}
    for dirpath, dirnames, _ in os.walk(root):
        if "_delta_log" in dirnames:
            rel = os.path.relpath(dirpath, root)
            tables[rel] = dirpath
            dirnames.remove("_delta_log")
    return tables

microns_tables = discover_delta_tables(MICRONS_ROOT)
visp_tables = discover_delta_tables(VISP_ROOT)
all_rel_paths = sorted(set(microns_tables) | set(visp_tables))

table_sources: dict[str, list[str]] = {}
for rel in all_rel_paths:
    sources = []
    if rel in microns_tables:
        sources.append(microns_tables[rel])
    if rel in visp_tables:
        sources.append(visp_tables[rel])
    table_sources[rel] = sources

print(f"Discovered {len(all_rel_paths)} delta tables")

In [ ]:
def combine_tables(rel_path: str, source_paths: list[str]) -> pl.DataFrame:
    """Read and concatenate delta table(s) from source_paths. Returns Polars DataFrame."""
    tables = []
    for src in source_paths:
        tables.append(DeltaTable(src).to_pyarrow_table())
    if len(tables) == 1:
        return pl.from_arrow(tables[0])
    return pl.from_arrow(pa.concat_tables(tables, promote_options="permissive"))

## 1. Create Taxonomy records

In [4]:
taxonomies = [
    Taxonomy(
        id="tasic_2018_visp_scrnaseq",
        name="Tasic 2018 VISp scRNA-seq T-types",
        description="Cell type taxonomy from Tasic et al. 2018 scRNA-seq of mouse VISp",
        publication="doi.org/10.1038/s41586-018-0654-5",
        project_id="tasic_visp_scrnaseq",
    ),
    Taxonomy(
        id="visp_met_types",
        name="VISp MET-types",
        description="Morpho-electric-transcriptomic cell types for mouse VISp",
        project_id="visp_patchseq",
    ),
    Taxonomy(
        id="minnie65",
        name="Minnie65 CSM clustering",
        description="Cell type clustering from Minnie65 EM volume dendrite ultrastructure",
        project_id="minnie65",
    ),
]

for t in taxonomies:
    print(f"  {t.id:35s}  project_id={t.project_id}")

  tasic_2018_visp_scrnaseq             project_id=tasic_visp_scrnaseq
  visp_met_types                       project_id=visp_patchseq
  minnie65                             project_id=minnie65


In [5]:
schema = build_arrow_schema(Taxonomy)
table = models_to_table(taxonomies, schema)
table = attach_linkml_metadata(table, linkml_class="Taxonomy")

os.makedirs(OUTPUT_ROOT, exist_ok=True)
taxonomy_path = os.path.join(OUTPUT_ROOT, "taxonomy")
write_deltalake(taxonomy_path, table, mode="overwrite", partition_by=["project_id"])

print(f"Wrote {table.num_rows} taxonomy records to {taxonomy_path}")
table

Wrote 3 taxonomy records to /scratch/combined_with_taxonomies/taxonomy


pyarrow.Table
project_id: string not null
id: string not null
name: string not null
description: string
publication: string
version: string
root: string
clusters: list<item: string>
  child 0, item: string
----
project_id: [["tasic_visp_scrnaseq","visp_patchseq","minnie65"]]
id: [["tasic_2018_visp_scrnaseq","visp_met_types","minnie65"]]
name: [["Tasic 2018 VISp scRNA-seq T-types","VISp MET-types","Minnie65 CSM clustering"]]
description: [["Cell type taxonomy from Tasic et al. 2018 scRNA-seq of mouse VISp","Morpho-electric-transcriptomic cell types for mouse VISp","Cell type clustering from Minnie65 EM volume dendrite ultrastructure"]]
publication: [["doi.org/10.1038/s41586-018-0654-5",null,null]]
version: [[null,null,null]]
root: [[null,null,null]]
clusters: [[null,null,null]]

## 2. Map cluster project_id to taxonomy id

The existing `cluster` table uses `project_id` as the de facto taxonomy namespace.
We create an explicit mapping so we can add a `taxonomy` column.

In [6]:
cluster_project_to_taxonomy = {
    "tasic_2018_visp_scrnaseq": "tasic_2018_visp_scrnaseq",
    "visp_met_types": "visp_met_types",
    "minnie65": "minnie65",
}

membership_project_to_taxonomy = {
    "visp_inh_patchseq": "visp_met_types",
    "visp_exc_patchseq": "visp_met_types",
    "minnie65": "minnie65",
}

celltoclustermapping_mappingset_to_taxonomy = {
    "visp_inh_patchseq_ttype_mapping": "tasic_2018_visp_scrnaseq",
    "visp_exc_patchseq_ttype_mapping": "tasic_2018_visp_scrnaseq",
    "visp_exc_wnm_mettype_mapping": "visp_met_types",
}

## 3. Add taxonomy column to cluster table

In [7]:
cluster_df = combine_tables("cluster", table_sources["cluster"])
print(f"Loaded cluster table: {cluster_df.shape}")
print(f"Unique project_ids: {cluster_df['project_id'].unique().to_list()}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/combined_datasets'

In [ ]:
cluster_df = cluster_df.with_columns(
    pl.col("project_id").replace(cluster_project_to_taxonomy).alias("taxonomy")
)

out_path = os.path.join(OUTPUT_ROOT, "cluster")
write_deltalake(out_path, cluster_df.to_arrow(), mode="overwrite", partition_by=["project_id"])
print(f"Wrote cluster table with taxonomy column: {cluster_df.shape}")
cluster_df.head()

## 4. Add taxonomy column to clustermembership table

In [ ]:
cm_df = combine_tables("clustermembership", table_sources["clustermembership"])
print(f"Loaded clustermembership table: {cm_df.shape}")
print(f"Unique project_ids: {cm_df['project_id'].unique().to_list()}")

In [ ]:
cm_df = cm_df.with_columns(
    pl.col("project_id").replace(membership_project_to_taxonomy).alias("taxonomy")
)

out_path = os.path.join(OUTPUT_ROOT, "clustermembership")
write_deltalake(out_path, cm_df.to_arrow(), mode="overwrite", partition_by=["project_id"])
print(f"Wrote clustermembership table with taxonomy column: {cm_df.shape}")
cm_df.head()

## 5. Add target_taxonomy column to celltoclustermapping table

In [ ]:
ccm_df = combine_tables("celltoclustermapping", table_sources["celltoclustermapping"])
print(f"Loaded celltoclustermapping table: {ccm_df.shape}")
print(f"Unique mapping_sets: {ccm_df['mapping_set'].unique().to_list()}")

In [ ]:
ccm_df = ccm_df.with_columns(
    pl.col("mapping_set").replace(celltoclustermapping_mappingset_to_taxonomy).alias("target_taxonomy")
)

out_path = os.path.join(OUTPUT_ROOT, "celltoclustermapping")
write_deltalake(out_path, ccm_df.to_arrow(), mode="overwrite", partition_by=["project_id"])
print(f"Wrote celltoclustermapping table with target_taxonomy column: {ccm_df.shape}")
ccm_df.head()

## 6. Combine and write remaining tables

In [ ]:
def discover_delta_tables(root: str) -> dict[str, str]:
    root = os.path.abspath(root)
    tables = {}
    for dirpath, dirnames, _ in os.walk(root):
        if "_delta_log" in dirnames:
            rel = os.path.relpath(dirpath, root)
            tables[rel] = dirpath
            dirnames.remove("_delta_log")
    return tables

In [ ]:
already_written = {"cluster", "clustermembership", "celltoclustermapping", "taxonomy"}

for rel_path in sorted(table_sources):
    if rel_path in already_written:
        continue

    df = combine_tables(rel_path, table_sources[rel_path])
    arrow_table = df.to_arrow()

    dt = DeltaTable(table_sources[rel_path][0])
    partition_cols = dt.metadata().partition_columns

    out_path = os.path.join(OUTPUT_ROOT, rel_path)
    os.makedirs(out_path, exist_ok=True)

    write_kwargs = dict(mode="overwrite")
    if partition_cols:
        write_kwargs["partition_by"] = partition_cols

    write_deltalake(out_path, arrow_table, **write_kwargs)
    print(f"  combined {rel_path:50s} {arrow_table.num_rows:>10,d} rows")

print("\nDone.")

## 7. Verification

Read back the output tables and print shapes to confirm.

In [ ]:
output_tables = discover_delta_tables(OUTPUT_ROOT)

print(f"{'Table':<50s} {'Rows':>10s} {'Cols':>6s}")
print("-" * 68)

for rel_path in sorted(output_tables):
    out_path = os.path.join(OUTPUT_ROOT, rel_path)
    df = pl.read_delta(out_path)
    print(f"{rel_path:<50s} {df.shape[0]:>10,d} {df.shape[1]:>6d}")

In [ ]:
print("Taxonomy table:")
pl.read_delta(os.path.join(OUTPUT_ROOT, "taxonomy"))

In [ ]:
print("Cluster table (sample with taxonomy column):")
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "cluster"))
df.select(["id", "project_id", "taxonomy"]).head(10)

In [ ]:
print("ClusterMembership table (sample with taxonomy column):")
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "clustermembership"))
df.select(["item", "cluster", "project_id", "taxonomy"]).head(10)

In [ ]:
print("CellToClusterMapping table (sample with target_taxonomy column):")
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "celltoclustermapping"))
df.select(["source_cell", "target_cluster", "mapping_set", "project_id", "target_taxonomy"]).head(10)